In [1]:
#CIA2 - 2460601 (A Deepak)
#Load Dataset
import pandas as pd
import numpy as np
df=pd.read_csv('paper_moisture_control.csv')
df

,timestamp,machine_id,batch_id,paper_grade,moisture_pct,temperature_C,humidity_pct,cycle_start,cycle_end,deviation_flag
0,2025-09-01 07:00:00,M004,BATCH_001,coated,6.229510,28.048834,56.041926,2025-09-01 07:00:00,2025-09-01 09:00:00,0
1,2025-09-01 07:10:00,M004,BATCH_001,coated,6.525639,24.146738,54.039701,2025-09-01 07:00:00,2025-09-01 09:00:00,0
2,2025-09-01 07:20:00,M004,BATCH_001,coated,6.528640,22.919657,54.670740,2025-09-01 07:00:00,2025-09-01 09:00:00,0
3,2025-09-01 07:30:00,M004,BATCH_001,coated,6.439983,29.414033,33.000346,2025-09-01 07:00:00,2025-09-01 09:00:00,0
4,2025-09-01 07:40:00,M004,BATCH_001,coated,7.116764,23.824238,45.115315,2025-09-01 07:00:00,2025-09-01 09:00:00,0
...,...,...,...,...,...,...,...,...,...,...
1295,2025-09-17 20:20:00,M001,BATCH_100,coated,6.822513,23.943807,46.186320,2025-09-17 19:00:00,2025-09-17 21:00:00,0
1296,2025-09-17 20:30:00,M001,BATCH_100,coated,6.698634,23.938234,NaN,2025-09-17 19:00:00,2025-09-17 21:00:00,0
1297,2025-09-17 20:40:00,M001,BATCH_100,NaN,6.695514,26.023287,48.055718,2025-09-17 19:00:00,2025-09-17 21:00:00,0
1298,2025-09-17 20:50:00,M001,BATCH_100,coated,5.969523,27.106910,48.089398,2025-09-17 19:00:00,2025-09-17 21:00:00,0


In [15]:
#Convert timestamps to datetime objects
time_cols = ['timestamp', 'cycle_start', 'cycle_end']
for col in time_cols:
    df[col] = pd.to_datetime(df[col])  # <--- This line must be indented
print(df[time_cols].dtypes)

timestamp      datetime64[ns]
cycle_start    datetime64[ns]
cycle_end      datetime64[ns]
dtype: object


In [4]:
#NumPy-based stats
moisture_arr = df['moisture_pct'].to_numpy()
avg_moisture = np.mean(moisture_arr)
var_moisture = np.var(moisture_arr)
print(f"Dataset Shape: {df.shape}")
print(f"Average Moisture: {avg_moisture:.4f}%")
print(f"Moisture Variance: {var_moisture:.4f}")

Dataset Shape: (1300, 10)
Average Moisture: nan%
Moisture Variance: nan


In [7]:
#(Adding a small noise factor to a copy of data to simulate sensor jitter)
np.random.seed(42)
synthetic_noise = np.random.normal(0, 0.05, size=len(df))
df['moisture_synthetic'] = df['moisture_pct'] + synthetic_noise
df['moisture_synthetic']

0       6.254346
1       6.518725
2       6.561024
3       6.516134
4       7.105057
          ...   
1295    6.817568
1296    6.744588
1297    6.681000
1298    5.982892
1299    6.648256
Name: moisture_synthetic, Length: 1300, dtype: float64

In [14]:
#2.Timestamp Alignment
#Aligning sensor readings
df['is_aligned'] = (df['timestamp'] >= df['cycle_start']) & (df['timestamp'] <= df['cycle_end'])
misaligned_count = (~df['is_aligned']).sum()
print(f"Rows with timestamp misalignment: {misaligned_count}")

Rows with timestamp misalignment: 0


In [10]:
df_clean = df[df['is_aligned']].copy()
df_clean

,timestamp,machine_id,batch_id,paper_grade,moisture_pct,temperature_C,humidity_pct,cycle_start,cycle_end,deviation_flag,moisture_synthetic,is_aligned
0,2025-09-01 07:00:00,M004,BATCH_001,coated,6.229510,28.048834,56.041926,2025-09-01 07:00:00,2025-09-01 09:00:00,0,6.254346,True
1,2025-09-01 07:10:00,M004,BATCH_001,coated,6.525639,24.146738,54.039701,2025-09-01 07:00:00,2025-09-01 09:00:00,0,6.518725,True
2,2025-09-01 07:20:00,M004,BATCH_001,coated,6.528640,22.919657,54.670740,2025-09-01 07:00:00,2025-09-01 09:00:00,0,6.561024,True
3,2025-09-01 07:30:00,M004,BATCH_001,coated,6.439983,29.414033,33.000346,2025-09-01 07:00:00,2025-09-01 09:00:00,0,6.516134,True
4,2025-09-01 07:40:00,M004,BATCH_001,coated,7.116764,23.824238,45.115315,2025-09-01 07:00:00,2025-09-01 09:00:00,0,7.105057,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1295,2025-09-17 20:20:00,M001,BATCH_100,coated,6.822513,23.943807,46.186320,2025-09-17 19:00:00,2025-09-17 21:00:00,0,6.817568,True
1296,2025-09-17 20:30:00,M001,BATCH_100,coated,6.698634,23.938234,NaN,2025-09-17 19:00:00,2025-09-17 21:00:00,0,6.744588,True
1297,2025-09-17 20:40:00,M001,BATCH_100,NaN,6.695514,26.023287,48.055718,2025-09-17 19:00:00,2025-09-17 21:00:00,0,6.681000,True
1298,2025-09-17 20:50:00,M001,BATCH_100,coated,5.969523,27.106910,48.089398,2025-09-17 19:00:00,2025-09-17 21:00:00,0,5.982892,True


In [16]:
# 3. Data Cleansing and Imputation
initial_count = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f"Duplicates removed: {initial_count - len(df_clean)}")

Duplicates removed: 0


In [18]:
#Impute missing moisture readings (Forward-fill and Interpolation)
df_clean.loc[df_clean.sample(frac=0.01, random_state=42).index, 'moisture_pct'] = np.nan
df_clean['moisture_pct'] = df_clean['moisture_pct'].ffill().interpolate(method='linear')
df_clean['moisture_pct'] 

0       6.229510
1       6.525639
2       6.528640
3       6.439983
4       7.116764
          ...   
1295    6.822513
1296    6.698634
1297    6.695514
1298    5.969523
1299    6.632171
Name: moisture_pct, Length: 1300, dtype: float64